# Step 6 (RQ1) — Adapter Placement Study on CIFAR-FS  (Kaggle)

Compares **post_pool** (Step-4 baseline, reused) vs **serial** vs **parallel**
placement of the *same* 1×1 Bottleneck adapter inside a frozen ResNet-18, for
both heads (evidential + softmax) → 4 new `results/phase3_placement_*` JSONs.

**Kaggle setup (do this first):**
1. **Settings → Accelerator → GPU T4**, and **Settings → Internet → On**.
2. **Add Data** → attach your `bpeft-data` dataset (holding
   `cifar-100-python.tar.gz` + `test_32x32.mat`). TinyImageNet downloads at runtime.
3. Run top to bottom. Results land in `/kaggle/working/thesis/results`; hit
   **Save Version** to keep them.

Runtime: the 4 configs are close to Bottleneck's cost (no Full-FT here), so ~30–60 min on a T4.

## 0. GPU + environment check

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch :', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU T4'

## 1. Clone the repo + install deps

Set `BRANCH` to the branch that has the Step 6 placement code. Needs Internet ON.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'                      # <-- branch with the Step 6 commit
REPO_DIR = '/kaggle/working/thesis'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
!pip -q install -r requirements.txt
assert os.path.exists('src/adapters/placement.py'), \
    'MISSING src/adapters/placement.py — push the Step 6 commit to this BRANCH.'
print('Step 6 placement code present — good.')

## 2. Stage data from the attached Kaggle Dataset

Kaggle-native equivalent of the Colab Drive-staging: copy the archives the
loaders short-circuit on, so nothing downstream hits the slow cs.toronto.edu
host. CIFAR-100 + SVHN come from the `bpeft-data` dataset; TinyImageNet (~240 MB)
downloads at runtime (Internet ON — Kaggle's link to Stanford is fast).

In [ ]:
import os, shutil, hashlib, glob

os.makedirs('data/svhn', exist_ok=True)

def _find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

# CIFAR-100 tarball (md5-checked) -> data/
CIFAR_MD5 = 'eb9058c3a382ffc7106e4002c42a8d85'
dst = 'data/cifar-100-python.tar.gz'
if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
    src = _find('cifar-100-python.tar.gz')
    assert src, ('cifar-100-python.tar.gz not found under /kaggle/input — '
                 'attach the bpeft-data dataset via Add Data.')
    shutil.copy(src, dst)
    md5 = hashlib.md5(open(dst, 'rb').read()).hexdigest()
    assert md5 == CIFAR_MD5, f'md5 mismatch: {md5} != {CIFAR_MD5}'
    print(f'[cifar] staged {dst}  (md5 OK, {os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print(f'[cifar] already present at {dst}')

# SVHN test_32x32.mat -> data/svhn/
dst = 'data/svhn/test_32x32.mat'
if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
    src = _find('test_32x32.mat')
    assert src, 'test_32x32.mat not found under /kaggle/input — add it to bpeft-data.'
    shutil.copy(src, dst)
    print(f'[svhn] staged {dst}  ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print(f'[svhn] already present at {dst}')

# TinyImageNet: stage from the dataset if you added the zip, else download at runtime.
tin = _find('tiny-imagenet-200.zip')
if tin:
    shutil.copy(tin, 'data/tiny-imagenet-200.zip')
    print('[tin] staged tiny-imagenet-200.zip from the dataset')
else:
    print('[tin] not in dataset — will download at runtime (Internet ON)')

## 3. Build the frozen CIFAR-FS Bertinetto split

`data/` is gitignored, so materialize the canonical 64/16/20 split once.

In [ ]:
!python scripts/build_cifar_fs_split.py
import json
sp = json.load(open('data/cifar_fs_split.json'))
print('split status:', sp.get('_status'))
print('sizes:', {k: len(v) for k, v in sp.items() if isinstance(v, list)})
assert sp.get('_status') != 'synthetic_fallback', 'split is the SYNTHETIC fallback — fix before running'

## 4. (optional) Tests

Confirm the new placement code passes before burning GPU.

In [ ]:
!python -u -m pytest -q tests/test_placement.py
# full suite (slower):
# !python -u -m pytest -q

## 5. Run the 4 placement configs (train + 600-episode eval)

serial/parallel × evidential/softmax. Already-finished configs are skipped, so
you can re-run this cell to resume. post_pool is reused from Step 4.5 (no run).

In [ ]:
import subprocess, os

NUM_EPISODES = 600
RUNS = [
    ('exp_phase3_placement_serial_evidential',   'serial'),
    ('exp_phase3_placement_serial_softmax',      'serial'),
    ('exp_phase3_placement_parallel_evidential', 'parallel'),
    ('exp_phase3_placement_parallel_softmax',    'parallel'),
]

def head_desc(name):
    return 'prototype-evidential' if name.endswith('evidential') else 'prototype-softmax'

def result_path(placement, name):
    return f'results/phase3_placement_{placement}_bottleneck_{head_desc(name)}_metrics.json'

def run(cmd):
    print('>>>', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

status = {}
for name, placement in RUNS:
    out = result_path(placement, name)
    if os.path.exists(out):
        print(f'== SKIP {name} (found {out}) =='); status[name] = 'skip (exists)'; continue
    print(f'\n{"="*72}\n== {name} ==\n{"="*72}', flush=True)
    cfg = f'configs/{name}.yaml'
    rc = run(['python', 'scripts/train.py', '--config', cfg, '--wandb-mode', 'disabled'])
    if rc != 0:
        status[name] = f'TRAIN failed (rc={rc})'; continue
    rc = run(['python', 'scripts/evaluate.py', '--config', cfg,
              '--num-episodes', str(NUM_EPISODES), '--wandb-mode', 'disabled',
              '--results-suffix', f'phase3_placement_{placement}', '--use-tinyimagenet'])
    status[name] = 'OK' if (rc == 0 and os.path.exists(out)) else f'EVAL failed (rc={rc})'

print('\n' + '=' * 40 + '\nRUN STATUS\n' + '=' * 40)
for name, _ in RUNS:
    print(f'  {name:44s} {status.get(name, "not run")}')

## 6. Summary table

The 4 placement runs alongside the reused post_pool (Step 4.5) baseline.

In [ ]:
import glob, json, os
import pandas as pd

rows = []
files = sorted(glob.glob('results/phase3_placement_*_metrics.json')) + \
        sorted(glob.glob('results/step45_bottleneck_*_metrics.json'))
for f in files:
    d = json.load(open(f)); base = os.path.basename(f)
    placement = ('post_pool' if base.startswith('step45')
                 else ('serial' if 'serial' in base else 'parallel'))
    rows.append({
        'placement': placement,
        'head'     : d.get('interpretation'),
        'n_params' : d.get('n_params'),
        'acc'      : round(d.get('accuracy_mean', float('nan')), 4),
        'f1'       : round(d.get('f1_macro_mean', float('nan')), 4),
        'ece'      : round(d.get('ece_pooled', float('nan')), 4),
        'ece_ts'   : round(d['ece_ts'], 4) if 'ece_ts' in d else None,
        'auroc_svhn': round(d.get('ood_auroc_mean', float('nan')), 4),
    })
df = pd.DataFrame(rows).sort_values(['placement', 'head']).reset_index(drop=True)
pd.set_option('display.width', 200); df

## 7. Comparison plot (RQ1)

In [ ]:
!python scripts/step6_placement_plot.py
from IPython.display import Image
Image('results/step6_placement_comparison.png')

## 8. Persist

Results are under `/kaggle/working/thesis/results` (kept as notebook output).
Click **Save Version** to snapshot. Download the 4 `phase3_placement_*` JSONs +
`step6_placement_comparison.png` to commit them back to the repo locally.